# Dynamic Pricing Strategy in Ride-Hailing
## A Growth vs Retention Decision Case

**Recommendation**: Zone-Governed Surge Pricing (1.75x / 2.0x / 2.5x caps)

**Structure**
- WS1: Demand Suppression & Customer Response
- WS2: Price Elasticity Estimation
- WS3: Unit Economics & LTV Stress Test
- WS4: Scenario Simulation
- Strategy: Options Evaluation, Porter's, Implementation, Investor Narrative

In [ ]:
import sys, io, os, warnings
sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8', errors='replace')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from numpy.polynomial import polynomial as P

plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11, 
                     'axes.titlesize': 13, 'axes.titleweight': 'bold'})

BASE = os.path.join('..')
CHARTS = os.path.join(BASE, 'charts')
os.makedirs(CHARTS, exist_ok=True)

SURGE_ORDER = ['1.0x', '1.25x', '1.5x', '1.75x', '2.0x', '2.5x']
SURGE_VALS  = [1.0, 1.25, 1.5, 1.75, 2.0, 2.5]
COLORS      = ['#2A9D8F', '#3DBAA5', '#457B9D', '#5A9BD5', '#E63946', '#C1121F']

ADJ_BASE_COMPLETION = 0.97  # 97% adjusted (data has 100% artifact at 1.0x)

### Load & Engineer Data

In [ ]:
df = pd.read_csv(os.path.join(BASE, 'data', 'cleaned_data.csv'))
df['surge_band'] = pd.Categorical(df['surge_band'], categories=SURGE_ORDER, ordered=True)
df['cancel_int'] = 1 - df['ride_completed_int']
df['cost_per_trip'] = df['platform_revenue_inr'] - df['contribution_margin_inr']
df['cm_per_request'] = df['contribution_margin_inr'] * df['ride_completed_int']

total_cm = df['contribution_margin_inr'].sum()
total_rev = df['platform_revenue_inr'].sum()
total_trips = len(df)
n_customers = df['customer_id'].nunique()

print(f'Dataset: {total_trips:,} trips  |  {n_customers:,} customers  |  3 cities')
print(f'Avg fare: Rs.{df["fare_inr"].mean():.1f}  |  Avg CM/trip: Rs.{df["contribution_margin_inr"].mean():.2f}')

---
## WORKSTREAM 1 — Demand Suppression & Customer Response

In [ ]:
comp_by_surge = df.groupby('surge_band')['ride_completed_int'].mean()
sat_by_surge  = df.groupby('surge_band')['satisfaction_score'].mean()
cancel_by_s   = df.groupby('surge_band', observed=True)['cancel_int'].mean()
cm_by_surge   = df.groupby('surge_band')['contribution_margin_inr'].mean()

ws1 = pd.DataFrame({'completion': comp_by_surge, 'cancel': cancel_by_s, 
                     'satisfaction': sat_by_surge, 'cm_per_trip': cm_by_surge,
                     'n_trips': df.groupby('surge_band').size()}).round(3)
ws1['cm_per_request'] = (ws1['cm_per_trip'] * ws1['completion']).round(2)
ws1['pct_trips'] = (ws1['n_trips'] / total_trips * 100).round(1)
ws1['cm_of_total_pct'] = (df.groupby('surge_band')['contribution_margin_inr'].sum() / total_cm * 100).round(1)
print(ws1.to_string())

**Key finding**: Satisfaction drops from 4.14 at 1.0x to 2.36 at 2.5x (cliff at ~1.5x). Cancellation rises from 0% to 18%. 62.8% of total CM comes from surge trips (31.2% of volume).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot(SURGE_VALS, sat_by_surge.values, 'o-', color='#E63946', lw=2.5, ms=8)
ax.fill_between(SURGE_VALS, sat_by_surge.values, alpha=0.15, color='#E63946')
ax.axvline(x=1.5, color='red', ls=':', alpha=0.6, label='Satisfaction cliff (~1.5x)')
ax.set_xlabel('Surge Multiplier'); ax.set_ylabel('Avg Satisfaction')
ax.set_title('Customer Satisfaction vs Surge'); ax.legend(); ax.grid(alpha=0.3)
ax.set_ylim(2, 4.5)

ax = axes[1]
x = range(6); w = 0.35
ax.bar(x, comp_by_surge.values, w, label='Completion', color='#2A9D8F', alpha=0.85)
ax.bar([i+w for i in x], cancel_by_s.values, w, label='Cancellation', color='#E63946', alpha=0.85)
ax.set_xticks([i+w/2 for i in x]); ax.set_xticklabels(SURGE_ORDER)
ax.set_ylabel('Rate'); ax.set_title('Completion & Cancellation by Surge'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[2]
cm_req = ws1['cm_per_request'].values
ax.bar(range(6), cm_req, color=COLORS, width=0.6, edgecolor='white')
ax.set_xticks(range(6)); ax.set_xticklabels(SURGE_ORDER)
ax.set_ylabel('CM/Request (Rs.)'); ax.set_title('CM per Ride Request'); ax.grid(alpha=0.3)
for i, v in enumerate(cm_req): ax.text(i, v+0.3, f'Rs.{v:.1f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(CHARTS, 'notebook_ws1_deterioration.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Negative margin analysis
pct_neg = df['is_negative_margin'].mean()
neg_margin = df.pivot_table('is_negative_margin', index='distance_bucket', columns='surge_band', aggfunc='mean')
surge_cm = df[df['surge_multiplier'] > 1.0]['contribution_margin_inr'].sum()
nonsurge_cm = df[df['surge_multiplier'] == 1.0]['contribution_margin_inr'].sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.pie([surge_cm, nonsurge_cm], labels=['Surge (62.8%)', 'Non-Surge (37.2%)'], 
        colors=['#E63946','#2A9D8F'], autopct='%1.1f%%', startangle=90)
ax1.set_title('CM Source')

im = ax2.imshow(neg_margin.values, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
ax2.set_xticks(range(6)); ax2.set_xticklabels(neg_margin.columns)
ax2.set_yticks(range(len(neg_margin.index))); ax2.set_yticklabels(neg_margin.index)
ax2.set_xlabel('Surge Band'); ax2.set_ylabel('Distance')
ax2.set_title(f'% Trips with Negative CM ({pct_neg:.0%} overall)')
for i in range(len(neg_margin.index)):
    for j in range(6):
        v = neg_margin.values[i,j]
        if not np.isnan(v): ax2.text(j,i,f'{v:.0%}',ha='center',va='center',fontsize=7,fontweight='bold',color='white' if v>0.5 else 'black')
plt.colorbar(im, ax=ax2, label='% Negative')
plt.tight_layout(); plt.show()

**Finding**: 39.6% of trips have negative CM — concentrated at short-distance (<3km) + 1.0x surge. These are NOT loss-leaders (r=-0.324, p<0.001 — no correlation with future revenue). 62.8% of CM comes from surge trips.

---
## WORKSTREAM 2 — Price Elasticity Estimation

In [ ]:
elast_results = pd.read_csv(os.path.join(BASE, 'outputs', 'elasticity_results.csv'))
elast_zone    = pd.read_csv(os.path.join(BASE, 'outputs', 'elasticity_by_zone.csv'))
elast_time    = pd.read_csv(os.path.join(BASE, 'outputs', 'elasticity_by_time_segment.csv'))
elast_sens    = pd.read_csv(os.path.join(BASE, 'outputs', 'elasticity_sensitivity_baseline.csv'))

avg_e = elast_results['elasticity'].mean()
e97 = elast_sens[elast_sens['adjusted_baseline_completion'] == 0.97]['arc_elasticity'].mean()
print(f'Arc elasticity (100% artifact baseline): {avg_e:.3f}')
print(f'Arc elasticity (97% realistic baseline):  {e97:.3f}')
print(f'Demand is INELASTIC ({e97:.3f} < 1.0) — 10% price increase → only {(e97*10):.1f}% demand loss')
print(f'\nElasticity by zone:\n', elast_zone.groupby('zone_type')['arc_elasticity'].mean().to_string())
print(f'\nElasticity by time segment:\n', elast_time.to_string(index=False))

In [ ]:
# Sensitivity to baseline completion assumption
fig, ax = plt.subplots(figsize=(8, 4.5))
baselines = elast_sens['adjusted_baseline_completion'].unique()
for b in sorted(baselines):
    sub = elast_sens[elast_sens['adjusted_baseline_completion'] == b]
    ax.plot(sub['surge_band'].str.replace('x',''), sub['arc_elasticity'], 'o-', label=f'Baseline={b:.0%}')
ax.set_xlabel('Surge Band'); ax.set_ylabel('Arc Elasticity')
ax.set_title('Elasticity Sensitivity to Baseline Completion Assumption'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Methodology**: Arc elasticity = Δlog(completed_demand) / Δlog(fare). Adjusted for 97% baseline completion (data has 100% artifact at 1.0x). Elasticity is inelastic everywhere — residential zones are most price-sensitive, transit hubs least.

---
## WORKSTREAM 3 — Unit Economics & Customer LTV

In [ ]:
# Churn model: satisfaction → annual churn (industry benchmarks)
def annual_churn(sat):
    if sat <= 1.5: return 0.55
    elif sat <= 2.0: return 0.45
    elif sat <= 2.5: return 0.38
    elif sat <= 3.0: return 0.28
    elif sat <= 3.5: return 0.20
    elif sat <= 4.0: return 0.14
    elif sat <= 4.5: return 0.10
    else: return 0.08

def monthly_churn(annual): return 1 - (1 - annual) ** (1/12)

# DCF LTV: 18-month horizon, 10% annual discount
MONTHLY_DISCOUNT = (1.10) ** (1/12) - 1

def dcf_ltv(arpu, cm_pct, monthly_churn_rate, months=18):
    retention, ltv = 1, 0
    for t in range(1, months+1):
        retention *= (1 - monthly_churn_rate)
        ltv += arpu * cm_pct * retention / (1 + MONTHLY_DISCOUNT) ** t
    return ltv

# Build customer profiles
cust = df.groupby('customer_id').agg(
    avg_satisfaction=('satisfaction_score', 'mean'),
    total_cm=('contribution_margin_inr', 'sum'),
    total_rev=('platform_revenue_inr', 'sum'),
    cac=('cac_inr', 'first'),
    segment=('customer_segment', 'first'),
).reset_index()

cust['annual_churn'] = cust['avg_satisfaction'].apply(annual_churn)
cust['monthly_churn'] = cust['annual_churn'].apply(monthly_churn)
cust['monthly_arpu'] = cust['total_cm'] / 12
cust['cm_pct'] = (cust['total_cm'] / cust['total_rev'].replace(0, np.nan)).fillna(0).clip(-5, 1)
cust['LTV'] = cust.apply(lambda r: dcf_ltv(r['monthly_arpu'], r['cm_pct'], r['monthly_churn']), axis=1)
cust['LTV_CAC'] = cust['LTV'] / cust['cac']

# Segment summary
seg_ltv = cust.groupby('segment').agg(
    avg_LTV=('LTV','mean'), avg_CAC=('cac','mean'),
    avg_churn=('annual_churn','mean'), avg_satisfaction=('avg_satisfaction','mean'),
    n_customers=('customer_id','count')
).round(2)
seg_ltv['LTV_CAC'] = (seg_ltv['avg_LTV'] / seg_ltv['avg_CAC']).round(2)
print(seg_ltv.to_string())

In [ ]:
# Churn sensitivity: LTV/CAC under increasing churn
sens_rows = []
for seg in ['frequent','occasional','rare']:
    s = seg_ltv.loc[seg]
    arpu = cust[cust['segment']==seg]['monthly_arpu'].mean()
    cm_pct = cust[cust['segment']==seg]['cm_pct'].mean()
    for label, mult in [('Base',1.0), ('+25%',1.25), ('+50%',1.50), ('+100%',2.0)]:
        adj = s['avg_churn'] * mult
        ltv = dcf_ltv(arpu, cm_pct, monthly_churn(adj))
        sens_rows.append({'segment':seg,'scenario':label,'annual_churn':round(adj,3),
                         'LTV':round(ltv,1),'CAC':round(s['avg_CAC'],1),'LTV_CAC':round(ltv/s['avg_CAC'],2)})
sens_df = pd.DataFrame(sens_rows)
print(sens_df.to_string(index=False))

In [ ]:
# NRR: Revenue-weighted retention
cust_trip_count = df.groupby('customer_id').size().reset_index(name='n_trips')
cust_trip_count['churned'] = (df.groupby('customer_id')['days_since_last_ride'].max().values > 30).astype(int)

total_customer_rev = cust['total_rev'].sum()
retained_rev = cust.loc[cust_trip_count['churned'] == 0, 'total_rev'].sum()
nrr = retained_rev / total_customer_rev * 100

customer_retention = 1 - cust_trip_count['churned'].mean()
print(f'Revenue-weighted NRR (Net Revenue Retention): {nrr:.1f}%')
print(f'Customer retention rate: {customer_retention:.1%}')
print(f'Reason: churned customers are low-value rare segment → revenue NRR >> customer retention')

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

segments = ['frequent','occasional','rare']
x = range(3)
ax = axes[0]
bars = ax.bar(x, seg_ltv.loc[segments, 'LTV_CAC'].values, color=['#2A9D8F','#457B9D','#E63946'], width=0.5)
ax.set_xticks(x); ax.set_xticklabels(['Frequent','Occasional','Rare'])
ax.set_ylabel('LTV/CAC Ratio'); ax.set_title('LTV/CAC by Segment')
ax.axhline(y=1.0, color='orange', ls='--', label='Break-even'); ax.legend(); ax.grid(alpha=0.3)
for b, v, s in zip(bars, seg_ltv.loc[segments,'LTV_CAC'].values, segments):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.03, f'{v:.2f}x', ha='center', fontweight='bold')

ax = axes[1]
for seg, c in zip(segments, ['#2A9D8F','#457B9D','#E63946']):
    sub = sens_df[sens_df['segment']==seg]
    ax.plot(sub['scenario'], sub['LTV_CAC'], 'o-', label=seg.title(), color=c, lw=2)
ax.set_ylabel('LTV/CAC'); ax.set_title('Churn Sensitivity'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CHARTS, 'notebook_ltv_cac.png'), dpi=150, bbox_inches='tight')
plt.show()

**Note**: Churn model uses industry benchmarks (sat → annual churn). Validation against behavioral proxy shows WEAK correlation — satisfaction → churn causation NOT confirmed in this dataset. Churn improvement magnitude is uncertain (0.15-0.64pp range). Recommendation rests on **structural thesis** (CM diversification + regulatory positioning), not churn improvement alone.

---
## WORKSTREAM 4 — Scenario Simulation (Trip-Level Methodology)

In [ ]:
# Interpolation functions with 97% adjusted completion baseline
def interp_completion(s): return float(np.interp(s, SURGE_VALS, comp_by_surge.values))
def interp_satisfaction(s): return float(np.interp(s, SURGE_VALS, sat_by_surge.values))
def adj_completion(s):
    if s <= 1.0: return ADJ_BASE_COMPLETION
    return max(0, min(1, ADJ_BASE_COMPLETION + (interp_completion(s) - interp_completion(1.0))))

# 97% elasticity by surge band
e_by_band = dict(zip(e97['surge_band'].str.replace('x',''), e97['arc_elasticity']))
avg_e97 = np.mean(list(e_by_band.values()))
e_by_band = {key: value for key, value in zip(SURGE_ORDER, [avg_e97]*6)}  # Use average for simplicity

# Scenarios (the 3 given options + combined + status quo)
scenarios = {
    'S0_Status_Quo':       {'desc': 'Status Quo',                                    'type': 'none'},
    'S1_Calibrated':       {'desc': 'Option 1: Calibrated Surge (2.0x)',             'type': 'uniform', 'cap': 2.0},
    'S2b_Zone_Moderate':   {'desc': 'Option 2: Zone Gov (1.75/2.0/2.5x)',           'type': 'zone',
                            'residential': 1.75, 'commercial': 2.0, 'transit_hub': 2.5},
    'S3a_Loyalty_Hard':    {'desc': 'Option 3: Loyalty Hard (1.5x Gold/Silver)',     'type': 'loyalty', 'loyalty_cap': 1.5},
    'S3b_Loyalty_Soft':    {'desc': 'Option 3: Loyalty Soft (1.75x Gold/Silver)',    'type': 'loyalty', 'loyalty_cap': 1.75},
    'S4b_Combined_Soft':   {'desc': 'Combined: Zone + Loyalty Soft (1.75x)',         'type': 'combined',
                            'residential': 1.75, 'commercial': 2.0, 'transit_hub': 2.5, 'loyalty_cap': 1.75},
}

In [ ]:
# TRIP-LEVEL SIMULATION ENGINE
# For each scenario: mark capped trips → compute CM/revenue/completion/satisfaction deltas
# Methodology: per-trip fixed cost structure, elasticity-based demand adjustment,
#             97% adjusted completion baseline, customer-level satisfaction aggregation

def run_simulation(sname, sparms):
    if sparms['type'] == 'none':
        return {'scenario': sname, 'desc': sparms['desc'], 'cm_change_pct': 0.0, 'rev_change_pct': 0.0,
                'cancel_change_pp': 0.0, 'satisfaction_change': 0.0, 'trips_capped_pct': 0.0,
                'n_customers_improved': 0, 'pct_customers_improved': 0.0, 'avg_churn_delta_pp': 0.0,
                'annual_cm_delta': 0, 'ltv_retention_gain': 0, 'net_annual_value': 0,
                'baseline': f'{ADJ_BASE_COMPLETION:.0%} (adjusted)'}

    t = df[['trip_id','customer_id','zone_type','surge_multiplier','surge_band',
            'fare_inr','platform_revenue_inr','contribution_margin_inr',
            'cost_per_trip','platform_take_rate','ride_completed_int',
            'cancel_int','satisfaction_score','loyalty_imputed']].copy()
    t['is_capped'] = False; t['effective_cap'] = np.nan

    if sparms['type'] == 'uniform':
        mask = t['surge_multiplier'] > sparms['cap']
        t.loc[mask, ['is_capped','effective_cap']] = [True, sparms['cap']]
    elif sparms['type'] == 'zone':
        for z in ['residential','commercial','transit_hub']:
            zmask = (t['zone_type'] == z) & (t['surge_multiplier'] > sparms[z])
            t.loc[zmask, ['is_capped','effective_cap']] = [True, sparms[z]]
    elif sparms['type'] == 'loyalty':
            loyal = t['loyalty_imputed'].isin(['Gold','Silver'])
            mask = loyal & (t['surge_multiplier'] > sparms['loyalty_cap'])
            t.loc[mask, ['is_capped','effective_cap']] = [True, sparms['loyalty_cap']]
    elif sparms['type'] == 'combined':
        for z in ['residential','commercial','transit_hub']:
            zmask = t['zone_type'] == z
            loyal = t['loyalty_imputed'].isin(['Gold','Silver'])
            eff = min(sparms['loyalty_cap'], sparms[z])
            t.loc[zmask & loyal & (t['surge_multiplier'] > eff), ['is_capped','effective_cap']] = [True, eff]
            t.loc[zmask & ~loyal & (t['surge_multiplier'] > sparms[z]), ['is_capped','effective_cap']] = [True, sparms[z]]

    capped = t['is_capped']
    n_capped = capped.sum()

    # CM & Revenue deltas (per-trip)
    cm_delta_total = rev_delta_total = cancel_delta_sum = 0.0
    if n_capped > 0:
        ct = t[capped].copy()
        ratio = ct['effective_cap'] / ct['surge_multiplier']
        ct['new_revenue'] = ct['platform_revenue_inr'] * ratio
        ct['new_cm'] = ct['new_revenue'] - ct['cost_per_trip']
        ct['new_completion'] = ct['effective_cap'].apply(adj_completion)
        ct['new_cancel'] = ct['effective_cap'].apply(interp_completion)
        ct['new_cancel'] = ct['effective_cap'].apply(lambda s: float(np.interp(s, SURGE_VALS, cancel_by_s.values)))

        # Demand elasticity effect
        price_reduction = (ct['surge_multiplier'] - ct['effective_cap']) / ct['surge_multiplier']
        ct['demand_increase'] = avg_e97 * price_reduction

        cm_delta_total = ((ct['new_cm'] * ct['new_completion'] * (1 + ct['demand_increase']))
                         - (ct['contribution_margin_inr'] * ct['ride_completed_int'])).sum()
        rev_delta_total = ((ct['new_revenue'] * ct['new_completion'] * (1 + ct['demand_increase']))
                          - (ct['platform_revenue_inr'] * ct['ride_completed_int'])).sum()
        cancel_delta_sum = (ct['new_cancel'] - ct['cancel_int']).sum()

    # Customer-level satisfaction & churn
    t['new_satisfaction'] = t['satisfaction_score'].copy()
    if n_capped > 0:
        capped_trips = t[capped].copy()
        capped_trips['new_satisfaction'] = capped_trips['effective_cap'].apply(interp_satisfaction)
        t.loc[capped, 'new_satisfaction'] = capped_trips['new_satisfaction'].values

    cust_new = t.groupby('customer_id')['new_satisfaction'].mean().reset_index()
    cust_new.columns = ['customer_id', 'new_avg_sat']
    cs = cust[['customer_id','monthly_arpu','cm_pct','avg_satisfaction','LTV','annual_churn']].merge(cust_new, on='customer_id')
    cs['new_churn'] = cs['new_avg_sat'].apply(annual_churn)
    cs['churn_delta'] = cs['annual_churn'] - cs['new_churn']  # positive = improvement
    cs['new_LTV'] = cs.apply(lambda r: dcf_ltv(r['monthly_arpu'], r['cm_pct'], monthly_churn(r['new_churn'])), axis=1)

    ltv_gain = (cs['new_LTV'] - cs['LTV']).sum()
    n_improved = (cs['churn_delta'] > 0).sum()
    annual_cm_delta = cm_delta_total * 12

    return {
        'scenario': sname, 'desc': sparms['desc'],
        'cm_change_pct': round(cm_delta_total / total_cm * 100, 2),
        'rev_change_pct': round(rev_delta_total / total_rev * 100, 2),
        'cancel_change_pp': round(cancel_delta_sum / total_trips * 100, 2),
        'satisfaction_change': round((t['new_satisfaction'].mean() - t['satisfaction_score'].mean()), 3),
        'trips_capped_pct': round(n_capped / total_trips * 100, 1),
        'n_customers_improved': n_improved,
        'pct_customers_improved': round(n_improved / n_customers * 100, 1),
        'avg_churn_delta_pp': round(cs['churn_delta'].mean() * 100, 2),
        'annual_cm_delta': round(annual_cm_delta),
        'ltv_retention_gain': round(ltv_gain),
        'net_annual_value': round(annual_cm_delta + ltv_gain),
        'baseline': f'{ADJ_BASE_COMPLETION:.0%} (adjusted)'
    }

print('Running trip-level simulation...')
sim_results = []
for sname, sparms in scenarios.items():
    r = run_simulation(sname, sparms)
    sim_results.append(r)
    print(f'  {sname}: CM={r["cm_change_pct"]:+.2f}%  Rev={r["rev_change_pct"]:+.2f}%  Churn={r["avg_churn_delta_pp"]:+.2f}pp  Trips={r["trips_capped_pct"]:.1f}%')

sim_df = pd.DataFrame(sim_results)

In [ ]:
# Results table
SIM_COLS = ['desc','cm_change_pct','rev_change_pct','cancel_change_pp','trips_capped_pct',
            'pct_customers_improved','avg_churn_delta_pp','annual_cm_delta','ltv_retention_gain','net_annual_value']
print(sim_df[SIM_COLS].to_string(index=False))

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
active = sim_df[sim_df['scenario'] != 'S0_Status_Quo'].copy()
labels = [r['desc'] for _, r in active.iterrows()]
x = range(len(active))

ax = axes[0]
bars = ax.bar(x, active['cm_change_pct'], color=['#FF6B35','#2E86AB','#A23B72','#F18F01','#6C5B7B'], width=0.6, edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=8)
ax.set_ylabel('CM Change (%)'); ax.set_title('CM Impact by Strategy'); ax.axhline(0,color='gray',ls='--',alpha=0.5); ax.grid(alpha=0.3)
for b, v in zip(bars, active['cm_change_pct']): ax.text(b.get_x()+b.get_width()/2, b.get_height()-0.3, f'{v:.1f}%', ha='center', va='top', fontweight='bold', color='white')

ax = axes[1]
bars = ax.bar(x, active['avg_churn_delta_pp'], color=COLORS[:5], width=0.6, edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=8)
ax.set_ylabel('Churn Improvement (pp)'); ax.set_title('Retention Improvement'); ax.grid(alpha=0.3)
for b, v in zip(bars, active['avg_churn_delta_pp']): ax.text(b.get_x()+b.get_width()/2, v+0.02, f'{v:.2f}', ha='center', fontweight='bold')

ax = axes[2]
ax.barh([l.replace('(','\n(') for l in labels], -active['cm_change_pct'], color='#E63946', alpha=0.7)
ax.set_xlabel('|CM Loss| (%)'); ax.set_title('Absolute CM Impact'); ax.grid(alpha=0.3)
for i, (c, t) in enumerate(zip(active['cm_change_pct'], active['trips_capped_pct'])):
    ax.text(abs(c)+0.3, i, f'{abs(c):.1f}% (capped {t:.0f}%)', va='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(CHARTS, 'notebook_simulation.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## STRATEGIC OPTIONS EVALUATION

In [ ]:
# Multi-criteria scoring
scoring = []
for _, r in active.iterrows():
    cm_score = max(1, min(5, 5 + r['cm_change_pct'] / 3))
    cancel_score = max(1, min(5, 3 + abs(r['cancel_change_pp']) * 2))
    churn_score = max(1, min(5, 3 + r['avg_churn_delta_pp'] * 5))

    if 'Zone' in r['desc']:
        reg_score = 5  # regulatory
    elif 'Loyalty' in r['desc']:
        reg_score = 3
    else:
        reg_score = 2

    if 'Moderate' in r['desc'] or 'Soft' in r['desc']:
        impl_score = 4
    else:
        impl_score = 2

    if 'Zone' in r['desc']:
        strat_score = 5
    elif 'Loyalty' in r['desc'] and 'Soft' in r['desc']:
        strat_score = 3
    else:
        strat_score = 2

    weights = {'CM':0.15,'Cancel':0.10,'Churn':0.15,'Reg':0.20,'Impl':0.15,'Strat':0.25}
    scores = {'CM':cm_score,'Cancel':cancel_score,'Churn':churn_score,'Reg':reg_score,'Impl':impl_score,'Strat':strat_score}
    weighted = sum(scores[k]*weights[k] for k in weights)
    scoring.append({'Option': r['desc'], **scores, 'Weighted': round(weighted, 2)})

score_df = pd.DataFrame(scoring).sort_values('Weighted', ascending=False)
print(score_df.to_string(index=False))

In [ ]:
# Radar chart
cats = ['CM','Cancel','Churn','Reg','Impl','Strat']
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2*np.pi, len(cats), endpoint=False).tolist() + [0]
for idx, (_, row) in enumerate(score_df.iterrows()):
    vals = [row[c] for c in cats] + [row[cats[0]]]
    ax.plot(angles, vals, 'o-', lw=2, label=row['Option'][:40], color=COLORS[idx], ms=6)
    ax.fill(angles, vals, alpha=0.1, color=COLORS[idx])
ax.set_xticks(angles[:-1]); ax.set_xticklabels(cats); ax.set_ylim(0,5.5)
ax.set_title('Strategy Option Scoring', y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.3,1.1), fontsize=8)
plt.tight_layout(); plt.show()

---
## COMPETITIVE ANALYSIS — Porter's Five Forces

In [ ]:
porters = pd.read_csv(os.path.join(BASE, 'outputs', 'porters_five_forces.csv'))
print(porters.to_string(index=False))

pricing_def = pd.read_csv(os.path.join(BASE, 'outputs', 'pricing_defensibility.csv'))
print('\nPricing Defensibility:')
print(pricing_def.to_string(index=False))

---
## IMPLEMENTATION ROADMAP

In [ ]:
impl = pd.read_csv(os.path.join(BASE, 'outputs', 'implementation_plan.csv'))
print(impl.to_string(index=False))

print('\nRisk Registry:')
risks = pd.read_csv(os.path.join(BASE, 'outputs', 'risk_registry.csv'))
print(risks.to_string(index=False))

---
## INVESTOR NARRATIVE

In [ ]:
inv = pd.read_csv(os.path.join(BASE, 'outputs', 'investor_narrative.csv'))
print(inv.to_string(index=False))

---
## CONCLUSION — KEY NUMBERS

In [ ]:
best = score_df.iloc[0]
print('=' * 70)
print('FINAL RECOMMENDATION')
print('=' * 70)
print(f'Recommended: {best["Option"]}')
print(f'Weighted Score: {best["Weighted"]:.2f} / 5')
print()
print('CANONICAL NUMBERS')
print(f'  Cancellation: 0% at 1.0x –> 18% at 2.5x')
print(f'  Satisfaction:  4.14 at 1.0x –> 2.36 at 2.5x (cliff at ~1.5x)')
print(f'  Negative margin: {pct_neg:.0%} of trips')
print(f'  CM from surge: {surge_cm/(surge_cm+nonsurge_cm)*100:.1f}% (from {df[df["surge_multiplier"]>1.0].shape[0]/total_trips*100:.1f}% of volume)')
print(f'  Elasticity: {e97:.2f} (97% baseline) — INELASTIC')
print(f'  LTV/CAC: Frequent {seg_ltv.loc["frequent","LTV_CAC"]:.2f}x | Occasional {seg_ltv.loc["occasional","LTV_CAC"]:.2f}x | Rare {seg_ltv.loc["rare","LTV_CAC"]:.2f}x')
print(f'  NRR (revenue-weighted): {nrr:.1f}%')
print(f'  Churn model confidence: MEDIUM-LOW (industry benchmarks not validated in this dataset)')
print(f'  Zone Gov CM impact: {sim_df[sim_df["scenario"]=="S2b_Zone_Moderate"]["cm_change_pct"].values[0]:.2f}%')
print(f'  Zone Gov churn improvement: {sim_df[sim_df["scenario"]=="S2b_Zone_Moderate"]["avg_churn_delta_pp"].values[0]:.2f}pp')
print(f'  Min fare to eliminate negative CM: Rs.125/zone')
print(f'  Zone Gov wins under all weight sensitivity scenarios (6/6)')
print(f'  Recommendation rests on STRUCTURAL THESIS: CM diversification + regulatory positioning + competitive defensibility')
print()
print('DELIVERABLES')
print(f'  D3: Unit_Economics.xlsx  |  D4: Revenue_Simulation.xlsx  |  D5: Strategy_Deck.pptx/pdf')
print(f'  Charts: {len(os.listdir(CHARTS))} in charts/  |  Key numbers: key_numbers.csv')
print(f'  Source: cleaned_data.csv ({total_trips:,} rows)')